# Fetch GMW v3 Fiji product run data

In [ ]:
import os
import requests
import geopandas as gpd
from shapely.geometry import shape

API_BASE_URL = os.getenv(
    "CSDR_API_BASE_URL", "https://csdr.dev.oceandevelopmentdata.org/api/v0"
)
API_KEY = os.getenv("CSDR_API_KEY")

if not API_KEY:
    raise EnvironmentError("Set CSDR_API_KEY before running this notebook.")

HEADERS = {
    "x-api-key": API_KEY,
    "Accept": "application/json",
}

print(f"API base URL: {API_BASE_URL}")
print("API key loaded from CSDR_API_KEY")

In [ ]:
def api_get(path: str, params: dict | None = None, headers=HEADERS):
    url = f"{API_BASE_URL.rstrip('/')}/{path.lstrip('/')}"
    response = requests.get(url, headers=headers, params=params, timeout=60)
    response.raise_for_status()

    payload = response.json()
    if "data" not in payload:
        raise ValueError(f"API response missing 'data' field: {response.text}")

    return payload["data"]


def get_items(path: str, params: dict | None = None, headers=HEADERS):
    query = dict(params or {})
    page = int(query.get("page", 1))
    query.setdefault("size", 10)

    while True:
        query["page"] = page
        page_payload = api_get(path, query, headers=headers)

        page_count = int(page_payload.get("pageCount", 1))
        rows = page_payload.get("data", [])

        for item in rows:
            yield item

        if page >= page_count:
            break

        page += 1

In [ ]:
# Get a list of all the products
products = list(get_items("/product"))
print(f"Found {len(products)} products.")

In [ ]:
# Plot each product, and create a mapping of product ID to product metadata
products_id = {}
for product in products:
    print(f"ID: {product['id']}, Title: {product['name']}")
    products_id[product["id"]] = product

In [ ]:
# Grab the two GMW products for later
gmw_v3 = products_id["c3ebeeb6-83ec-4b13-8985-88178d643984"]
gmw_v4 = products_id["f7cf7d28-9e39-4e3c-8102-705fc3eb40a0"]

In [ ]:
# Get the geometry outputs (shared between products)
geometries = list(
    get_items(
        f"/geometries-run/{gmw_v3['mainRun']['geometriesRun']['id']}/outputs/export"
    )
)
print(len(geometries))

In [ ]:
# Convert to GeoDataFrame for plotting
rows = []
for item in geometries:
    rows.append(
        {
            "id": item.get("id"),
            "name": item.get("name"),
            **(item.get("properties") or {}),
            "geometry": shape(item["geometry"]),
        }
    )

gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
gdf.explore()

In [ ]:
# Get the values for the v3 and v4 GMW products for all geometries at a specific time point
params = {"timePoint": "2020-01-01T00:00:00.000Z"}

gmw_v3_outputs = list(
    get_items(f"/product-run/{gmw_v3['mainRunId']}/outputs/export", params=params)
)
gmw_v4_outputs = list(
    get_items(f"/product-run/{gmw_v4['mainRunId']}/outputs/export", params=params)
)

print(
    f"Found {len(gmw_v3_outputs)} outputs for GMW v3 and {len(gmw_v4_outputs)} outputs for GMW v4."
)

In [ ]:
# Append gmw_v3 and v4 'value' field to the geodataframe
gdf["gmw_v3"] = None
gdf["gmw_v4"] = None

for output in gmw_v3_outputs:
    geometry_id = output["geometryOutputId"]
    value = output["value"] / 10_000_000_000
    gdf.loc[gdf["id"] == geometry_id, "gmw_v3"] = value

for output in gmw_v4_outputs:
    geometry_id = output["geometryOutputId"]
    value = output["value"] / 10_000_000_000
    gdf.loc[gdf["id"] == geometry_id, "gmw_v4"] = value

# Drop where value is 0 or null in either v3 or v4
gdf = gdf[
    (gdf["gmw_v3"].notnull())
    & (gdf["gmw_v4"].notnull())
    & (gdf["gmw_v3"] != 0)
    & (gdf["gmw_v4"] != 0)
].copy()

# Add the difference
gdf["gmw_diff"] = gdf["gmw_v4"] - gdf["gmw_v3"]
gdf.head()

# Make sure the three values we care about are numeric
gdf["gmw_v3"] = gdf["gmw_v3"].astype(float).round(3)
gdf["gmw_v4"] = gdf["gmw_v4"].astype(float).round(3)
gdf["gmw_diff"] = gdf["gmw_diff"].astype(float).round(3)

# Relative diff
gdf["gmw_rel_diff"] = gdf["gmw_diff"] / gdf["gmw_v3"]

# Keep only name, gmw_v3, gmw_v4, gmw_diff, geometry
simple_gdf = gdf[["name", "gmw_v3", "gmw_v4", "gmw_diff", "geometry"]]

simple_gdf.head()

In [ ]:
# Plot the difference, just for the top 5
top_5 = gdf.nlargest(5, "gmw_diff")
ax = top_5.plot.scatter(
    x="gmw_v3",
    y="gmw_diff",
    c="gmw_diff",
    cmap="coolwarm",
    s=50,
    alpha=0.7,
    edgecolor="k",
)
for i, row in top_5.iterrows():
    ax.annotate(
        row["name"],
        (row["gmw_v3"], row["gmw_diff"]),
        textcoords="offset points",
        xytext=(5, 5),
        ha="left",
    )

In [ ]:
# Plot the relative difference, just for the top 5
top_5 = gdf.nlargest(5, "gmw_rel_diff")
ax = top_5.plot.scatter(
    x="gmw_v3",
    y="gmw_rel_diff",
    c="gmw_rel_diff",
    cmap="coolwarm",
    s=50,
    alpha=0.7,
    edgecolor="k",
)
for i, row in top_5.iterrows():
    ax.annotate(
        row["name"],
        (row["gmw_v3"], row["gmw_rel_diff"]),
        textcoords="offset points",
        xytext=(5, 5),
        ha="left",
    )

In [ ]:
# Make a pretty map
simple_gdf.explore(column="gmw_diff", cmap="RdYlGn", legend=False)

In [ ]:
# And a nice map of just the v3 values
simple_gdf.explore(column="gmw_v3", cmap="Greens", legend=False)